# DAEN 328 Assignment 5: Load and Serve

**Name:** _your name here_

Save this notebook as `A5_LastName_FirstName.ipynb`. It must run **top to bottom** (Kernel > Restart & Run All) with all outputs showing when you submit.

Work through `Assignment05_Guide.ipynb` first. Every technique you need is taught there on practice data. This notebook is where you apply it to `EmployeeDB`.

Your dashboard (Part C) is a separate file, `app.py`. See the instructions PDF.

## CONFIG: edit this cell once

In [37]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

DB_NAME = "EmployeeDB"
DB_USER = "postgres"
DB_PASSWORD = "Superuser2006"      # change to YOUR PostgreSQL password
DB_HOST = "localhost"
DB_PORT = "5432"

CSV_FILE = "cleaned_data.csv"   # your Assignment 3 output, in the same folder as this notebook

DB_PARAMS = {"dbname": DB_NAME, "user": DB_USER, "password": DB_PASSWORD,
             "host": DB_HOST, "port": DB_PORT}
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Part A: Load your cleaned CSV into the 7 EmployeeDB tables

### A1. Read and inspect

Read the CSV with every column as text. Show its shape and the number of blanks per column.

In [38]:
# TODO A1
#Read CSV with every column as text Show its shape and the number of blanks per column.
df = pd.read_csv(CSV_FILE, dtype=str)
print("Shape of the DataFrame:", df.shape)
print("Number of blanks per column:")
print(df.isnull().sum())

Shape of the DataFrame: (14320, 31)
Number of blanks per column:
EmployeeID                   0
Name                         0
HighestDegree                0
Division                     0
DivisionAddress              0
DivisionPhone                0
PersonalAddress              0
Email                        0
PhoneNumber                  0
Department                   0
DepartmentDescription       54
JobTitle                     1
JobDescription               1
Salary                       1
Birthdate                    0
SocialSecurity               0
Honorific                13591
First Name                   0
Middle Name                  0
Last Name                    0
Degree                   13608
DivStreetAddress             0
DivCity                      0
DivState                     0
DivZip                       0
Status                       0
PerStreetAddress             0
PerState                     0
PerZip                       0
PerCity                      0
Inval

### A2. Clean and validate in pandas

Required:
- Strip spaces from text columns.
- Convert `Birthdate` to a real date (`pd.to_datetime(..., errors="coerce").dt.date`) and **print how many could not be read**.
- Convert `Salary` and `EmployeeID` to numbers.
- Put every row that cannot satisfy your schema's NOT NULL or CHECK constraints into a `rejects` DataFrame, save it as `rejects_employees.csv`, and print rows read / good / rejected. Never invent a value (such as a salary of 0) to get a row past a constraint.

In [39]:
# TODO A2
# Strip whitespace and treat whitespace-only values as missing.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip().replace("", pd.NA)

# Birthdate is nullable, so count unreadable values without rejecting the row.
df["Birthdate"] = pd.to_datetime(df["Birthdate"], errors="coerce").dt.date
birthdate_unreadable = int(df["Birthdate"].isna().sum())
print("Birthdates that are blank or unreadable:", birthdate_unreadable)

# Convert numeric columns; invalid values become missing for validation below.
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
df["EmployeeID"] = pd.to_numeric(df["EmployeeID"], errors="coerce")

# Required by EmployeeDB to not be null: name, SSN, department, job title, division, and salary.
required_columns = ["Name", "SocialSecurity", "Department", "JobTitle", "Division", "Salary"]
missing_required = df[required_columns].isna().any(axis=1)
invalid_ssn = df["SocialSecurity"].duplicated()
invalid_salary = df["Salary"].isna() | df["Salary"].le(0)
invalid = missing_required | invalid_salary | invalid_ssn



good = df.loc[~invalid].copy()
rejects = df.loc[invalid].copy()
rejects.to_csv("rejects_employees.csv", index=False)
print(f"Rows read: {len(df)}   good: {len(good)}   rejected: {len(rejects)}")

Birthdates that are blank or unreadable: 0
Rows read: 14320   good: 14318   rejected: 2


### Given: `clean()` (run it, do not change it)

Pass **every** value through `clean()` when you insert it. pandas marks a blank as `NaN`, and if a `NaN` reaches PostgreSQL it is stored as the value `NaN`, not as NULL. A `NaN` salary passes `NOT NULL` and `CHECK (salary > 0)` with no error, and then every average that includes it comes out as `NaN`. `clean()` turns blanks into `None` (a real NULL), so the database rejects what it should. It also turns numpy numbers into plain Python numbers, which avoids the error `can't adapt type numpy.int64`.

In [40]:
def clean(value):
    # Turn pandas blanks into None (SQL NULL) and numpy numbers into plain Python numbers.
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

### Given: start over with empty tables

While you build the loader, a bad attempt can leave wrong rows behind. To empty all 7 tables and restart the id numbers from 1, set `RESET = True`, run the cell, then set it back to `False`. This keeps your tables and constraints; it only removes the rows. Do not use `DROP TABLE`, which deletes your Assignment 4 schema.

Leave `RESET = False` when you submit.

In [41]:
RESET = True

if RESET:
    with engine.begin() as conn:
        conn.execute(text("""
            TRUNCATE employeedetails, personaldetails, jobdescriptions, divisionlocations,
                     departments, jobtitles, divisions
            RESTART IDENTITY CASCADE;
        """))
    print("All 7 tables emptied.")

All 7 tables emptied.


### A3. The loader

Write `load_employees(good)`. Requirements:
- Loads all 7 tables, lookup tables first, `employeedetails` last, using the CSV-to-column map in the instructions PDF.
- Runs in **one transaction** (all or nothing).
- **Safe to re-run**: a second run adds zero rows to every table.
- Uses `%s` placeholders for every value, and passes every value through `clean()`.

In [42]:
# TODO A3
def load_employees(good):
    conn = psycopg2.connect(**DB_PARAMS)
    try:
        with conn:                                   # one transaction: all or nothing
            with conn.cursor() as cur:

                # ---- Pattern 1: lookup tables, DB generates the id ----
                for name in good["Division"].unique():
                    cur.execute(
                        "INSERT INTO divisions (divtype) VALUES (%s) "
                        "ON CONFLICT (divtype) DO NOTHING;",
                        (clean(name),))

                for title in good["JobTitle"].unique():
                    cur.execute(
                        "INSERT INTO jobtitles (jobtitle) VALUES (%s) "
                        "ON CONFLICT (jobtitle) DO NOTHING;",
                        (clean(title),))

                depts = (good[["Department", "DepartmentDescription"]]
                         .sort_values("DepartmentDescription", na_position="last")
                         .drop_duplicates("Department"))
                for _, r in depts.iterrows():
                    cur.execute(
                        "INSERT INTO departments (departmentname, departmentdescription) "
                        "VALUES (%s, %s) ON CONFLICT (departmentname) DO NOTHING;",
                        (clean(r["Department"]), clean(r["DepartmentDescription"])))

                # ---- Pattern 1b: read the generated ids back into dictionaries ----
                cur.execute("SELECT divtype, divisionid FROM divisions;")
                div_ids = dict(cur.fetchall())
                cur.execute("SELECT jobtitle, jobtitleid FROM jobtitles;")
                job_ids = dict(cur.fetchall())
                cur.execute("SELECT departmentname, departmentid FROM departments;")
                dept_ids = dict(cur.fetchall())

                # ---- Pattern 2: key comes from the CSV (ssn) ----
                people = good.drop_duplicates("SocialSecurity")
                for _, r in people.iterrows():
                    cur.execute(
                        "INSERT INTO personaldetails "
                        "(ssn, name, honorific, highestdegree, email, phonenumber, "
                        " birthdate, perstreet, percity, perstate, perzip) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) "
                        "ON CONFLICT (ssn) DO NOTHING;",
                        (clean(r["SocialSecurity"]), clean(r["Name"]), clean(r["Honorific"]),
                         clean(r["HighestDegree"]), clean(r["Email"]), clean(r["PhoneNumber"]),
                         clean(r["Birthdate"]), clean(r["PerStreetAddress"]),
                         clean(r["PerCity"]), clean(r["PerState"]), clean(r["PerZip"])))

                # ---- Pattern 3: child tables, no UNIQUE, so check with NOT EXISTS ----
                locations = good[["Division", "DivStreetAddress", "DivCity", "DivState",
                                  "DivZip", "DivisionPhone"]].drop_duplicates("Division")
                for _, r in locations.iterrows():
                    did = div_ids[r["Division"]]
                    cur.execute(
                        '''INSERT INTO divisionlocations
                               (divisionid, divstreet, divcity, divstate, divzip, divisionphone)
                           SELECT %s, %s, %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM divisionlocations
                                             WHERE divisionid = %s);''',
                        (did, clean(r["DivStreetAddress"]), clean(r["DivCity"]),
                         clean(r["DivState"]), clean(r["DivZip"]), clean(r["DivisionPhone"]),
                         did))

                jobs = good[["JobTitle", "Division", "JobDescription"]] \
                           .drop_duplicates(["JobTitle", "Division"])
                for _, r in jobs.iterrows():
                    jid = job_ids[r["JobTitle"]]
                    did = div_ids[r["Division"]]
                    cur.execute(
                        '''INSERT INTO jobdescriptions (jobtitleid, divisionid, description)
                           SELECT %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM jobdescriptions
                                             WHERE jobtitleid = %s AND divisionid = %s);''',
                        (jid, did, clean(r["JobDescription"]), jid, did))

                # ---- Pattern 4: fact table, foreign keys from the dictionaries ----
                for _, r in good.iterrows():
                    emp_id = clean(r["EmployeeID"])
                    emp_id = int(emp_id) if emp_id is not None else None
                    cur.execute(
                        '''INSERT INTO employeedetails
                               (employeeid, ssn, departmentid, jobtitleid, divisionid, salary)
                           SELECT %s, %s, %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM employeedetails
                                             WHERE ssn = %s);''',
                        ((emp_id), clean(r["SocialSecurity"]),
                         dept_ids[r["Department"]], job_ids[r["JobTitle"]],
                         div_ids[r["Division"]], clean(r["Salary"]),
                         clean(r["SocialSecurity"])))
    finally:
        conn.close()


### A4. Run it twice and reconcile

Run your loader, record the 7 table counts, run it **again**, record them again. Show one table with, for every table: the count you expected from the CSV, the count after run 1, the count after run 2, and whether all three agree.

Then run the given NaN check below it. Every count must be 0.

In [43]:
# TODO A4
import time
TABLES = ["divisions", "jobtitles", "departments", "personaldetails",
          "divisionlocations", "jobdescriptions", "employeedetails"]

def table_counts(tables=TABLES):
    with engine.connect() as conn:
        return {t: conn.execute(text(f"SELECT COUNT(*) FROM {t};")).scalar() for t in tables}

load_employees(good)
after_run_1 = table_counts()

load_employees(good)                      # second run: should add NOTHING
after_run_2 = table_counts()

expected = {
    "divisions":         good["Division"].nunique(),
    "jobtitles":         good["JobTitle"].nunique(),
    "departments":       good["Department"].nunique(),
    "personaldetails":   good["SocialSecurity"].nunique(),
    "divisionlocations": good["Division"].nunique(),                    # one location per division (matches loader)
    "jobdescriptions":   len(good[["JobTitle", "Division"]].drop_duplicates()),
    "employeedetails":   len(good),                                     # one row per good CSV row
}

recon = pd.DataFrame({"expected_from_csv": expected,
                      "after_run_1": after_run_1,
                      "after_run_2": after_run_2})
recon["ok"] = (recon["expected_from_csv"] == recon["after_run_1"]) & \
              (recon["after_run_1"] == recon["after_run_2"])
recon
time.perf_counter()

6733.085886

In [44]:
# Given: NaN check. Every count must be 0. If not, a value reached the database without clean().
with engine.connect() as conn:
    nan_check = {
        "salary = NaN":    conn.execute(text("SELECT COUNT(*) FROM employeedetails WHERE salary = 'NaN';")).scalar(),
        "name = 'NaN'":    conn.execute(text("SELECT COUNT(*) FROM personaldetails WHERE name = 'NaN';")).scalar(),
        "jobtitle = 'NaN'": conn.execute(text("SELECT COUNT(*) FROM jobtitles WHERE jobtitle = 'NaN';")).scalar(),
    }
nan_check

{'salary = NaN': 0, "name = 'NaN'": 0, "jobtitle = 'NaN'": 0}

### A5. Load report (write in this cell, 4 to 6 sentences)

- How many rows you read, loaded and rejected, and the reason for each kind of reject.
- Which loading pattern you used for `divisionlocations`, `jobdescriptions` and `employeedetails`, and why.
- One real problem you hit while building the loader (the error message or wrong count) and how you found and fixed it.


I read 14320 rows total, and only 2 were rejected. The only rejects I put were for nulls for columns that shouldn't be null in the schema. 

I picked pattern 3 and 4: Inserted into child table checking for duplicates becuase jobdescription and divsionlocation ren't set to unique so we have to avoid dupications. Also used fact table for employeedetails and cross checks with SSN

No errors whatsoever

# Part B: SQL

Write your own SQL for each question in the code cell under it, using the `q()` helper, and leave the result showing. Rules for every question:
- Filter by **names** (`'Engineering'`, `'Austin'`), never by an id number you looked up.
- Never type a number that the database can compute (an average, a count).
- Show names, not ids, wherever a person would read the result.

In [45]:
def q(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params)

**Q1.** Headcount by department: each department's name and number of employees, largest first.

In [46]:
# Q1
q('''
SELECT d.departmentname, COUNT(*) AS headcount
FROM employeedetails e
JOIN departments d ON d.departmentid = e.departmentid
GROUP BY d.departmentname
ORDER BY headcount DESC;
''')

,departmentname,headcount
0,Operations,10579
1,Administrative,2000
2,Engineering,1085
3,Human Resources,644
4,Management,5
5,Executive,5


**Q2.** Engineering employees earning less than $100,000: name, job title and salary, highest salary first.

In [47]:
# Q2
q('''
SELECT p.name, j.jobtitle, e.salary
FROM employeedetails e
JOIN personaldetails p ON p.ssn = e.ssn
JOIN jobtitles j ON j.jobtitleid = e.jobtitleid
JOIN departments d ON d.departmentid = e.departmentid
WHERE d.departmentname = 'Engineering'
  AND e.salary < 100000
ORDER BY e.salary DESC;
''')

,name,jobtitle,salary
0,Jennifer Bonnie Duffy,Engineer Level 4,99980.47
1,Luis Christopher Holder,Engineer Level 3,99890.86
2,Michelle Mary Davis,Engineer Level 1,99860.47
3,Lisa Crystal Austin,Engineer Level 3,99838.28
4,Catherine Deanna Hunter,Engineer Level 4,99820.89
...,...,...,...
391,Brian Shawn Parrish,Engineer Level 1,81480.18
392,Billy Daniel Jenkins,Engineer Level 1,81478.14
393,Kaitlyn Amanda Allen,Engineer Level 1,81471.96
394,Jeremiah Jared Marshall,Engineer Level 1,81424.51


**Q3.** Human Resources employees who work at the division located in **Austin**: name, job title and division. Find the division through its city; do not type the division's name.

In [48]:
# Q3
q('''
SELECT p.name, j.jobtitle, dv.divtype AS division
FROM employeedetails e
JOIN personaldetails p ON p.ssn = e.ssn
JOIN jobtitles j ON j.jobtitleid = e.jobtitleid
JOIN departments d ON d.departmentid = e.departmentid
JOIN divisions dv ON dv.divisionid = e.divisionid
WHERE d.departmentname = 'Human Resources'
  AND e.divisionid IN (SELECT divisionid
                       FROM divisionlocations
                       WHERE divcity = 'Austin')
ORDER BY p.name;
''')

,name,jobtitle,division
0,Alan Jordan Rivera,Human Resources Specialist 3,South Division
1,Alexandra Jody Reese,Human Resources Specialist 1,South Division
2,Andrew Rodney Williams,Human Resources Specialist 1,South Division
3,Angela Brenda Cook,Human Resources Specialist 4,South Division
4,Angela Megan Hernandez,Human Resources Manager Level 1,South Division
...,...,...,...
123,Victor Andrew Curry,Human Resources Manager Level 2,South Division
124,Walter Martin Hale,Human Resources Specialist 1,South Division
125,Wendy Amy Stokes,Human Resources Manager Level 1,South Division
126,Wendy Linda Reynolds,Human Resources Specialist 2,South Division


**Q4.** Human Resources only: for each division, the number of HR employees and their average salary (2 decimals), highest average first.

In [49]:
# Q4
q('''
SELECT dv.divtype AS division,
       COUNT(*) AS hr_employees,
       ROUND(AVG(e.salary), 2) AS avg_salary
FROM employeedetails e
JOIN departments d ON d.departmentid = e.departmentid
JOIN divisions dv ON dv.divisionid = e.divisionid
WHERE d.departmentname = 'Human Resources'
GROUP BY dv.divtype
ORDER BY avg_salary DESC;
''')

,division,hr_employees,avg_salary
0,East Division,129,53599.01
1,North Division,129,53200.76
2,West Division,129,53046.26
3,South Division,128,52930.12
4,Central Division,129,52919.29


**Q5.** Engineering employees whose salary is at or above the average Engineering salary: name, job title, salary. The average must be computed inside the query.

In [50]:
# Q5
q('''
SELECT p.name, j.jobtitle, e.salary
FROM employeedetails e
JOIN personaldetails p ON p.ssn = e.ssn
JOIN jobtitles j ON j.jobtitleid = e.jobtitleid
JOIN departments d ON d.departmentid = e.departmentid
WHERE d.departmentname = 'Engineering'
  AND e.salary >= (SELECT AVG(e2.salary)
                   FROM employeedetails e2
                   JOIN departments d2 ON d2.departmentid = e2.departmentid
                   WHERE d2.departmentname = 'Engineering')
ORDER BY e.salary DESC;
''')

,name,jobtitle,salary
0,Lindsey Breanna Villa,Lead Engineering Manager,291487.46
1,Timothy Matthew Ortega,Lead Engineering Manager,262471.31
2,Jillian Danielle Bray,Lead Engineering Manager,231262.00
3,Rodney Kyle Anderson,Lead Engineering Manager,223912.18
4,Samantha Nicole Snyder,Lead Engineering Manager,208455.58
...,...,...,...
456,Michael Jeffrey Petersen,Engineer Level 4,108514.87
457,Douglas Mark Anderson,Engineer Level 1,108401.88
458,Eric Michael Velez,Engineer Level 1,108376.49
459,Patricia Jennifer Miller,Engineer Level 1,108323.75


**Q6.** Employees aged 50 or older versus under 50 (age as of today): for each group, the number of employees and the average salary. Leave out anyone with no birthdate.

In [51]:
# Q6
q('''
SELECT CASE WHEN DATE_PART('year', AGE(CURRENT_DATE, p.birthdate)) >= 50
            THEN '50 or older' ELSE 'Under 50' END AS age_group,
       COUNT(*) AS employees,
       ROUND(AVG(e.salary), 2) AS avg_salary
FROM employeedetails e
JOIN personaldetails p ON p.ssn = e.ssn
WHERE p.birthdate IS NOT NULL
GROUP BY age_group
ORDER BY age_group;
''')

,age_group,employees,avg_salary
0,50 or older,3256,46922.32
1,Under 50,11062,43874.98


**Q7.** For each division: its city and state, its headcount and its average salary, largest headcount first.

In [52]:
# Q7
q('''
SELECT dv.divtype AS division,
       l.divcity AS city,
       l.divstate AS state,
       COUNT(*) AS headcount,
       ROUND(AVG(e.salary), 2) AS avg_salary
FROM employeedetails e
JOIN divisions dv ON dv.divisionid = e.divisionid
JOIN divisionlocations l ON l.divisionid = dv.divisionid
GROUP BY dv.divtype, l.divcity, l.divstate
ORDER BY headcount DESC;
''')

,division,city,state,headcount,avg_salary
0,South Division,Austin,TX,2867,45557.88
1,West Division,Sacramento,CA,2863,44343.46
2,North Division,Springfield,IL,2863,44347.41
3,East Division,Albany,NY,2863,44240.25
4,Central Division,Columbus,OH,2862,44349.34


**Q8.** Departments whose average salary is higher than the company-wide average salary, with that department average.

In [53]:
# Q8
q('''
SELECT d.departmentname,
       ROUND(AVG(e.salary), 2) AS dept_avg_salary
FROM employeedetails e
JOIN departments d ON d.departmentid = e.departmentid
GROUP BY d.departmentname
HAVING AVG(e.salary) > (SELECT AVG(salary) FROM employeedetails)
ORDER BY dept_avg_salary DESC;
''')

,departmentname,dept_avg_salary
0,Executive,756238.76
1,Management,227950.60
2,Engineering,108286.63
3,Human Resources,53139.41


**Q9.** The highest-paid employee in each department: department, name, job title and salary.

In [54]:
# Q9
q('''
SELECT departmentname, name, jobtitle, salary
FROM (
    SELECT d.departmentname, p.name, j.jobtitle, e.salary,
           RANK() OVER (PARTITION BY d.departmentid
                        ORDER BY e.salary DESC) AS rnk
    FROM employeedetails e
    JOIN personaldetails p ON p.ssn = e.ssn
    JOIN jobtitles j ON j.jobtitleid = e.jobtitleid
    JOIN departments d ON d.departmentid = e.departmentid
) ranked
WHERE rnk = 1
ORDER BY departmentname;
''')

,departmentname,name,jobtitle,salary
0,Administrative,Christopher Justin Stanley,Secretary Level 8,55967.39
1,Engineering,Lindsey Breanna Villa,Lead Engineering Manager,291487.46
2,Executive,Mark Alexander Hill,Chief Executive Officer,1111605.37
3,Human Resources,Michelle Angela Conley,Lead HR Manager,192030.36
4,Management,Darryl Taylor Heath,Division Manager,249261.08
5,Operations,Veronica Rebecca Mclaughlin,Lead Operations Manager,293491.71


**Q10.** Data quality: every email address that belongs to more than one person, with how many people share it. Then answer in the markdown cell below.

In [55]:
# Q10
q('''
SELECT email, COUNT(DISTINCT ssn) AS people_sharing
FROM personaldetails
WHERE email IS NOT NULL
GROUP BY email
HAVING COUNT(DISTINCT ssn) > 1
ORDER BY people_sharing DESC, email;
''')

,email,people_sharing
0,anthony_thomas_200@KLB.com,2
1,cheryl_peterson_139@KLB.com,2
2,christopher_young_182@KLB.com,2
3,james_smith_138@KLB.com,2
4,jessica_anderson_140@KLB.com,2
5,joseph_davis_144@KLB.com,2
6,kevin_jones_171@KLB.com,2
7,luis_davis_161@KLB.com,2
8,melissa_moore_147@KLB.com,2
9,michael_morgan_159@KLB.com,2


**Q10, part 2 (2 to 3 sentences):** Look at the rows Q10 returned. Is a shared email a data error, or could it be legitimate? KLB wants to use email as each employee's login. What would you check or change in the data or the schema before that is safe?

I think it not leigitamte as these are the emails of people not a group of people.
I would check the rows against the names and examine whether duplicate names exist or entries.

# Part C: Dashboard

Your dashboard is `app.py`, a separate file. Build it from the specification in the instructions PDF, run it with `streamlit run app.py`, and put your screenshots and answers in the PDF. Nothing for Part C goes in this notebook.

# Part D: Industry scenarios

These are the kind of incidents data engineers handle at large companies. There is no single right answer. Answer each one in the same cell, under the question, in 3 to 5 sentences. Refer to **your own** loader, SQL or `app.py` wherever you can: a specific answer (the line, the query, the setting) beats a general one.

**D1. The job said SUCCESS.** Your loader runs every night. For months it has rejected a handful of rows each night. One Monday it rejects about 4,100 of 14,000 rows, the job still reports SUCCESS, and the dashboard's headcount drops by almost a third. Nobody notices for two days. The cause: the HR system started sending salaries as `"$85,000.00"` instead of `85000`.

1. Look at how **your** loader converts `Salary`. What would it do with `"$85,000.00"`, and where would those rows end up? If your loader would behave differently from the story above, say how.
It would put it as nan and eventually be rejected from loading. The loader would need to now handle the rejected salary entries. A warning message or stop of the loading woud also be good to notify me of any changes.
2. Why did nothing fail?
Because my loader doesn't give warning when it drops a row, it just skips it and nothing else.

3. What would you add to the loader so that next time this stops the run or alerts someone?
Alerts for when the loader catches something that is close enough to warrant invstigation instead of just being dropped.

**D2. Two numbers for the same thing.** Finance's report and your dashboard both read the same `EmployeeDB`, but they show different average salaries for Engineering. Your manager asks which one is wrong.

1. Give three legitimate reasons the two numbers could differ without either query having a bug. 
Could be different due to data from different times, how my notebook handles missing values by imputing over department, and how my notebook also doesn't check for whether a salary is abnormally high or low to warrant incorrectness.
At least one reason must come from how your own `app.py` works.
2. How would you settle which number the company should use?
I would use mine over finances because its cleaned well for general case, but indivudal cases I would trust the finances department as the source.

**D3. From 14,000 rows to 50 million.** After KLB acquires another company, the nightly file grows from about 14,000 rows to 50 million.

1. Time one run of your loader on your own data (for example, wrap one of the runs in A4 with `time.perf_counter()`) and report the time.
About 6700 seconds. 

2. What in your loader would break or slow down first at 50 million rows? Point to the specific part of your code and explain why it gets slow.
Definently the "NOT EXIST" portions because it has to iterate over every SSN for example and cross check the new entry against all 50 million entered SSNs.

3. What would you change?
I would probably find some way to query without using SSN as the primary key. Maybe my loader would check entries with multiple primary keys of smaller volume such that it could quit earlier if found invald.